# 02 — Attention-head attribution

Notebook 01 localized the IOI circuit to a small cast of attention heads with per-head patching: a 12×12 grid where a handful of cells lit up. But patching only tells you *that* a head matters — it collapses each head into one number ("X% of the behavior restored") and says nothing about **what the head is doing**. A +12% head and a −29% head are both "important"; the grid can't tell you one *writes the answer* and the other *attends to the wrong name*.

This notebook opens those heads up. Three complementary tools, each answering a different question:

| tool | question it answers | mechanism |
|------|--------------------|-----------|
| **direct logit attribution (DLA)** | which heads write the answer *into the output*? | project each head's residual write onto the IO−S logit direction |
| **per-head patching** (from 01) | which heads matter *causally*, direct or indirect? | splice clean head output into the corrupted run |
| **attention patterns** | *where* does a head read from? | inspect the `[query, key]` attention weights |

The punchline you're about to recover: **DLA and patching disagree, and the disagreement is the whole circuit.** Heads that dominate DLA (write the answer) are mostly *silent* in patching, and heads that dominate patching are mostly *silent* in DLA. That's not a bug — it's the difference between the heads that *act* and the heads that *set up* the action. Two head families, two measurement tools, each blind to the other.

**This notebook is exercises, not a demo.** Same loop as 01: each exercise has a `TODO` cell, a checkpoint with numbers verified on this exact model + prompt (GPT-2 small, fp32), and a collapsed solution. Predict before you run.

| # | What I predicted | What happened |
|---|------------------|---------------|
| 2 | *(fill in before running)* | |
| 3 | | |
| 4 | | |
| 5 | | |
| 6 | | |


In [ ]:
import torch
from transformer_lens import HookedTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)          # interventions only, no training

model = HookedTransformer.from_pretrained("gpt2", device=device)
model.eval()
NL, NH = model.cfg.n_layers, model.cfg.n_heads
print(f"device: {device}  layers={NL}  heads={NH}  d_model={model.cfg.d_model}")

## Setup — the same clean/corrupted pair as 01

Nothing new here; we re-establish the prompts, the logit-diff metric, and both cached runs so this notebook stands alone. The corruption is identical: swap the second `John` (`S2`, position 10) for `Mary` so the answer flips.

Note the named positions we'll reach for:

- **`IO_POS` = 4** — the `" Mary"` token (indirect object, the answer to copy).
- **`S1_POS` = 2** — the first `" John"`.
- **`S2_POS` = 10** — the second `" John"` (the duplicated subject, and the corrupted token).
- **`END` = 14** — the final `" to"`, where the next-token prediction is read out.

In [ ]:
clean_prompt     = "When John and Mary went to the store, John gave a drink to"
corrupted_prompt = "When John and Mary went to the store, Mary gave a drink to"

io_tok = model.to_single_token(" Mary")   # correct answer on the clean prompt
s_tok  = model.to_single_token(" John")

clean_tokens     = model.to_tokens(clean_prompt)
corrupted_tokens = model.to_tokens(corrupted_prompt)
str_toks = model.to_str_tokens(clean_prompt)
n_pos = len(str_toks)

IO_POS, S1_POS, S2_POS, END = 4, 2, 10, n_pos - 1
print(list(enumerate(str_toks)))
print(f"IO_POS={IO_POS}({str_toks[IO_POS]!r})  S1_POS={S1_POS}  S2_POS={S2_POS}  END={END}")

def logit_diff(logits) -> float:
    return (logits[0, -1, io_tok] - logits[0, -1, s_tok]).item()

clean_logits,     clean_cache     = model.run_with_cache(clean_tokens)
corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_tokens)
CLEAN_LD = logit_diff(clean_logits)
CORR_LD  = logit_diff(corrupted_logits)

def ioi_metric(logits) -> float:
    return (logit_diff(logits) - CORR_LD) / (CLEAN_LD - CORR_LD)

print(f"clean_ld={CLEAN_LD:+.3f}  corrupted_ld={CORR_LD:+.3f}   (expect +3.172 / -3.535)")

The heatmap helper, copied from 01 (no plotting deps — a colored HTML table is plenty). Cells show whatever value you pass × 100; red = positive, blue = negative, white = zero. We'll reuse it for patching metrics, DLA, and attention weights alike.

In [ ]:
from IPython.display import HTML, display

def heatmap(grid, col_labels, row_labels, title="", scale=1.05):
    """grid: indexable as [row][col]. Values are auto-scaled to ~[-scale, scale]."""
    def color(v):
        x = max(min(float(v) / scale, 1.0), -1.0)
        if x >= 0:
            return f"rgb(255,{int(255 * (1 - 0.75 * x))},{int(255 * (1 - 0.85 * x))})"
        t = -x
        return f"rgb({int(255 * (1 - 0.75 * t))},{int(255 * (1 - 0.5 * t))},255)"
    th = "padding:2px 7px;font-size:11px"
    head = "<tr><th></th>" + "".join(f"<th style='{th}'>{c}</th>" for c in col_labels) + "</tr>"
    body = "".join(
        f"<tr><th style='{th};text-align:right'>{rl}</th>" + "".join(
            f"<td style='background:{color(v)};{th};text-align:right'>{float(v)*100:.0f}</td>"
            for v in row) + "</tr>"
        for rl, row in zip(row_labels, grid))
    display(HTML(f"<div style='font-family:monospace'><b>{title}</b>"
                 f"<table style='border-collapse:collapse'>{head}{body}</table></div>"))

## Exercise 1 — direct logit attribution per head

**The idea.** The final logits are `W_U` applied to the END residual stream (after the final LayerNorm). The residual stream is a *sum* — every head and MLP wrote into it additively. So the logit diff is **linear** in those contributions:

$$\text{logit\_diff} = (\text{resid}_{\text{END}}^{\text{normed}}) \cdot (W_U[:,\text{IO}] - W_U[:,\text{S}])$$

Call $d = W_U[:,\text{IO}] - W_U[:,\text{S}]$ the **logit-diff direction**: a single vector in `d_model` space such that *how much any residual-stream vector points along $d$* is exactly how much it pushes the answer toward Mary over John. Because the stream is additive, you can ask this of *each head's contribution separately* — that's DLA.

**Head $h$'s contribution to the END residual** is its output `z` mapped back to `d_model` through `W_O`:

```
head_out = z[END, h] @ W_O[L, h]          # [d_head] @ [d_head, d_model] -> [d_model]
```

Two subtleties that bite if you skip them:

1. **The final LayerNorm.** Logits are read *after* `ln_final`, which divides the stream by a per-position scale. In folded transformer-lens weights LN's learned gain is already absorbed into `W_U`, so the only thing left to apply is the **scale** — divide each head's contribution by `cache["ln_final.hook_scale"][0, END]`. Treating the scale as a frozen constant is the standard DLA approximation (it's near-constant across the heads' small perturbations).
2. **Centering.** LayerNorm subtracts the mean over `d_model`. Project the *mean-centered* contribution, or equivalently note that $d$ itself can be centered once. We'll center each contribution; it's a rounding-level effect here but conceptually correct.

**Your job:** fill `dla[L, h]` with head $(L,h)$'s contribution to the clean logit diff, then rank.

**Checkpoint — top positive heads (the answer-writers):**

```
9.9  ≈ +1.66    9.6  ≈ +1.32    10.0 ≈ +0.88    10.10 ≈ +0.74    8.10 ≈ +0.46
```

**and top negative heads (writing against the answer):**

```
10.7 ≈ -1.46    11.10 ≈ -1.10    11.1 ≈ -0.84    10.6 ≈ -0.54
```

In [ ]:
W_U = model.W_U                              # [d_model, d_vocab]
logit_diff_dir = W_U[:, io_tok] - W_U[:, s_tok]   # d : the IO-S direction in resid space

dla = torch.zeros(NL, NH)

# TODO: for each layer L, take clean_cache["z", L][0, END]  -> [head, d_head],
#   map through model.W_O[L] (einsum "hd,hdm->hm") to get [head, d_model],
#   divide by the ln_final scale at END, mean-center over d_model,
#   then dot with logit_diff_dir to fill dla[L].
...

heatmap(dla, [f"H{h}" for h in range(NH)], [f"L{i}" for i in range(NL)],
        "direct logit attribution per head (raw logit units)", scale=1.7)

flat = dla.flatten()
print("top positive (answer-writers):")
for i in flat.topk(5).indices:
    print(f"  {i.item()//NH}.{i.item()%NH}  {flat[i].item():+.3f}")
print("top negative (anti-answer):")
for i in flat.topk(4, largest=False).indices:
    print(f"  {i.item()//NH}.{i.item()%NH}  {flat[i].item():+.3f}")

<details><summary><b>Solution — Exercise 1</b></summary>

```python
scale = clean_cache["ln_final.hook_scale"][0, END]      # [1], the per-position LN scale

for L in range(NL):
    z = clean_cache["z", L][0, END]                     # [head, d_head]
    head_out = torch.einsum("hd,hdm->hm", z, model.W_O[L])   # [head, d_model]
    head_out = head_out / scale                         # undo the final-LN scaling
    head_out = head_out - head_out.mean(-1, keepdim=True)    # LN centering
    dla[L] = head_out @ logit_diff_dir
```

transformer-lens also exposes `cache.stack_head_results(...)` and `cache.apply_ln_to_stack(...)` which do exactly this vectorized; doing it by hand once is worth more than the convenience.
</details>

### Reading the DLA map

A completely different picture from the patching grid in 01. The big DLA heads are concentrated in **L9–L11**, and they come in two signs:

- **Positive — name movers: 9.9 (+1.66), 9.6 (+1.32), 10.0 (+0.88), 10.10 (+0.74).** These are the heads that *write the answer*: their output, projected onto the IO−S direction, pushes hard toward Mary. 9.9 alone accounts for ~half a logit-diff worth of signal.
- **Negative — negative name movers: 10.7 (−1.46), 11.10 (−1.10), 11.1 (−0.84).** These write *against* the answer, toward John. You met 10.7 in 01 as the −29% patching head; DLA confirms what it's doing — it directly suppresses the correct token.

The net of all this opposition is still positive (the movers win), which is why the model gets it right. But note: the **S-inhibition heads from 01's patching grid (7.3, 8.6, 8.10) are nearly absent here** — 8.6 is essentially zero in DLA. Hold that thought; exercise 3 explains it.

## Exercise 2 — DLA vs. patching, side by side

Re-run 01's per-head patching grid (it's cheap — 144 forward passes) and put it next to DLA. This is the core observation of the notebook, so make it concrete.

**Predict first** (log it up top): you have the DLA ranking and you saw 01's patching ranking. Will the same heads top both? If not, *which kind* of head tops each, and why would a head be big in one and tiny in the other?

**Checkpoint — patching top |metric|:** `10.7 −29.5`, `8.10 +19.5`, `8.6 +15.2`, `11.10 −12.3`, `9.9 +12.1`, `7.3 +11.9`, `5.5 +11.5`.

Now line up the two measurements for the same heads:

```
head    patch%     DLA          role
9.9     +12.1     +1.66    name mover         (big in BOTH)
9.6      +0.3     +1.32    name mover         (big DLA, ~0 patch)
8.6     +15.2     -0.04    S-inhibition       (big patch, ~0 DLA)
8.10    +19.5     +0.46    S-inhibition       (big patch, small DLA)
7.3     +11.9     +0.13    S-inhibition       (big patch, ~0 DLA)
10.7    -29.5     -1.46    negative mover     (big in BOTH, negative)
```

In [ ]:
headgrid = torch.zeros(NL, NH)

# TODO: 01's per-head patch. For each (L, H): splice clean_cache["z", L][:, :, H, :]
# into the corrupted run at f"blocks.{L}.attn.hook_z" (all positions), score ioi_metric.
...

heatmap(headgrid, [f"H{h}" for h in range(NH)], [f"L{i}" for i in range(NL)],
        "per-head z patching — % of behavior restored", scale=0.30)

# Compare the two rankings on the named heads.
print(f"{'head':<6}{'patch%':>9}{'DLA':>9}")
for (L, H) in [(9,9),(9,6),(10,0),(8,6),(8,10),(7,3),(7,9),(5,5),(10,7),(11,10)]:
    print(f"{L}.{H:<4}{headgrid[L,H].item()*100:>+9.1f}{dla[L,H].item():>+9.2f}")

<details><summary><b>Solution — Exercise 2</b></summary>

```python
def patch_head(layer, head):
    def hook(act, hook):                       # act: [batch, pos, head, d_head]
        act[:, :, head, :] = clean_cache["z", layer][:, :, head, :]
        return act
    logits = model.run_with_hooks(
        corrupted_tokens, fwd_hooks=[(f"blocks.{layer}.attn.hook_z", hook)])
    return ioi_metric(logits)

for L in range(NL):
    for H in range(NH):
        headgrid[L, H] = patch_head(L, H)
```
</details>

### Why the two disagree — direct vs. total effect

This is the load-bearing idea of the notebook, so state it precisely:

- **DLA measures the *direct* path** — head → residual stream → logits, with nothing in between. It only sees a head if that head writes onto the final logit-diff direction *itself*.
- **Patching measures the *total* causal effect** — direct path **plus every indirect path** that runs through downstream heads and MLPs. Patch a head's output and you change not only its own logit write but everything that *reads* from it later.

So the four cases in the comparison table fall out cleanly:

- **9.9 — big in both.** A name mover that writes the answer directly (big DLA) *and*, being late and load-bearing, has little above it to route through, so its direct effect ≈ its total effect (big patch).
- **9.6 — big DLA, ~0 patch.** Writes the answer (DLA +1.32) but is **redundant**: knock it out by patching and the backup movers (10.0, 10.10, …) compensate, so the *total* effect is ~0. This is the backup/hydra redundancy you falsified 9.9-necessity with in 01, caught quantitatively.
- **8.6, 7.3 — big patch, ~0 DLA.** These **don't write the answer at all** (DLA ≈ 0). Their entire effect is *indirect*: they feed the name movers. Patching sees it (the movers' input changes); DLA is blind to it (no direct logit write). These are the **S-inhibition heads**.
- **10.7 — big and negative in both.** A negative name mover: writes *against* the answer directly, and that direct write is most of its (negative) total effect.

The clean summary: **DLA finds the heads that act on the output; patching finds the heads that matter, including those that act only through other heads.** You need both lenses to see the whole circuit.

## Exercise 3 — attention patterns: where do the name movers read?

DLA told us 9.9/9.6/10.0 write the answer. *How* do they know the answer is "Mary"? The hypothesis from the circuit story: a name mover at the END position **attends to the IO token** and copies it. Time to look directly at the attention weights and check.

`cache["pattern", L]` has shape `[batch, head, query, key]`. We want the row `query = END` — "from the final position, how much does this head attend to each earlier token?" — and specifically the weight on `key = IO_POS` (Mary).

**Predict:** for a name mover, where should the END→? attention mass concentrate? And for the *negative* name mover 10.7 — does it attend somewhere *else* (reading the wrong name), or to the *same* place (reading the right name but writing it with the wrong sign)? The answer to that second question tells you what "negative mover" actually means mechanistically.

**Checkpoint — attention from END to IO(Mary), clean prompt:**

```
9.9  -> 0.70     9.6  -> 0.55     10.0 -> 0.45        (positive movers)
10.7 -> 0.76     11.10 -> 0.73                        (negative movers)
```

In [ ]:
# Per-head attention from END -> IO(Mary), as a grid.
attn_to_io = torch.zeros(NL, NH)
# TODO: fill attn_to_io[L, H] with cache["pattern", L][0, H, END, IO_POS]  (clean cache)
...

heatmap(attn_to_io, [f"H{h}" for h in range(NH)], [f"L{i}" for i in range(NL)],
        "attention from END -> IO token (Mary), clean prompt", scale=0.8)

# For the candidate movers, print the top-3 keys they attend to from END.
print("END attends to (top-3 keys):")
for (L, H) in [(9,9),(9,6),(10,0),(10,7),(11,10)]:
    patt = clean_cache["pattern", L][0, H, END]          # [key_pos]
    tk = patt.topk(3)
    tops = ", ".join(f"{str_toks[p.item()]!r}:{v.item():.2f}" for v, p in zip(tk.values, tk.indices))
    print(f"  {L}.{H}:  IO(Mary)={patt[IO_POS].item():.2f}   top3=[{tops}]")

<details><summary><b>Solution — Exercise 3</b></summary>

```python
for L in range(NL):
    attn_to_io[L] = clean_cache["pattern", L][0, :, END, IO_POS]
```

`pattern[batch, head, query, key]` — fix `batch=0`, `query=END`, `key=IO_POS`, sweep heads.
</details>

### What the patterns show

The name movers do exactly what the name says: **from the final position, they attend overwhelmingly to the IO token and copy it.**

- **9.9 puts 0.70 of its attention on " Mary"**, 9.6 puts 0.55, 10.0 puts 0.45. The rest of their mass sits mostly on the BOS token (`<|endoftext|>`), the standard attention "rest position." They barely look at " John" (9.9 → 0.11).
- **The negative movers 10.7 (0.76) and 11.10 (0.73) attend to Mary even *harder* than the positive ones.** This is the mechanistic meaning of "negative name mover": they read the *correct* IO token, but their `W_O @ W_U` path writes it into the logits with a *negative* sign — they copy-suppress. So 10.7 isn't confused about who the IO is; it's actively betting against it. (Why a trained model wants this is still open — best current guess is calibration / preventing overconfidence.)

So the answer-writing mechanism is: **END → IO attention → copy.** The only thing still unexplained is *why the movers attend to Mary and not John* — both are names sitting in earlier positions. That's the S-inhibition heads' job, next.

## Exercise 4 — S-inhibition: the heads that point the movers

The name movers attend to **Mary, not John** — but a priori both are equally-valid earlier name tokens. Something biases the movers' attention *away from the subject*. That's the S-inhibition heads (the big-patch / zero-DLA family from exercise 2: 7.3, 7.9, 8.6, 8.10).

The story: S-inhibition heads sit at END, **attend back to S2** (the duplicated subject, position 10), read "the subject is John / lives at this position," and write a signal into END's residual stream that the name movers' *query* then reads — steering the movers' attention off John and onto Mary. They never touch the logits directly (hence ~0 DLA); they operate entirely on the movers' attention.

**Part A — confirm they attend to S2.** Inspect each candidate's END→? attention.

**Checkpoint:**

```
8.6  -> S2(John): 0.89      7.9  -> S2: 0.56      8.10 -> S2: 0.54      7.3 -> S2: 0.16
```

(7.3 is the weakest of the four on this particular prompt — it spreads onto `" gave"` — but still tops the patching grid via the value it writes. Heads aren't always textbook-clean; report what you measure.)

In [ ]:
print("S-inhibition candidates — END attends to (top-3 keys):")
for (L, H) in [(7,3),(7,9),(8,6),(8,10)]:
    patt = clean_cache["pattern", L][0, H, END]
    tk = patt.topk(3)
    tops = ", ".join(f"{str_toks[p.item()]!r}:{v.item():.2f}" for v, p in zip(tk.values, tk.indices))
    print(f"  {L}.{H}:  S2(John)={patt[S2_POS].item():.2f}   top3=[{tops}]")

**Part B — the causal test: break S-inhibition, watch a name mover's attention move.** If S-inhibition really *steers the movers*, then **corrupting** the S-inhibition heads' contribution should make a name mover stop preferring Mary. The clean way to test this: run the *corrupted* prompt (where S2 is now "Mary", so S-inhibition fires on the wrong token), and read **9.9's attention to the IO position**. If S-inhibition is what aims 9.9, its attention to the true IO should drop relative to the clean run.

**Predict:** clean, 9.9 attends 0.70 to IO(Mary). On the corrupted prompt — where the second name is now Mary, so "the duplicated subject" *is* Mary and S-inhibition should push the movers *away* from Mary — what happens to 9.9's attention onto position 4?

**Checkpoint:** 9.9's attention to position 4 (the original IO slot) **collapses from 0.70 (clean) to ≈ 0.01 (corrupted)** — essentially total. The corruption flips which name is the duplicated subject, S-inhibition re-aims, and the mover almost entirely stops attending to position 4; its mass swings to the now-unduplicated John. The effect is far from subtle — the mover's attention is *contingent* on the upstream S-inhibition state, not fixed.

In [ ]:
# 9.9 attention from END onto position 4, clean vs corrupted.
clean_99  = clean_cache["pattern", 9][0, 9, END, IO_POS].item()
corr_99   = corrupted_cache["pattern", 9][0, 9, END, IO_POS].item()
print(f"head 9.9  END -> pos4  attention:")
print(f"  clean     (pos4 = ' Mary', IO):           {clean_99:.3f}")
print(f"  corrupted (pos4 = ' Mary', now NOT dup):  {corr_99:.3f}")
print(f"  shift: {corr_99 - clean_99:+.3f}   (name mover re-aimed by S-inhibition change)")

<details><summary><b>Solution — Exercise 4</b></summary>

Part A is just reading `pattern[0, H, END, S2_POS]` for each head — no TODO, run it. Part B compares the same attention weight across the two cached runs you already have:

```python
clean_99 = clean_cache["pattern", 9][0, 9, END, IO_POS].item()
corr_99  = corrupted_cache["pattern", 9][0, 9, END, IO_POS].item()
```

The collapse (0.70 → 0.01) is indirect evidence, not a clean single-head intervention — the *whole prompt* changed, not just S-inhibition. The rigorous version patches *only* the S-inhibition heads' output and re-reads 9.9's pattern (a "path patch"); that's notebook 03's territory. The point here is to *see* that the mover's attention is contingent on upstream state, not fixed.
</details>

### The circuit, assembled

You've now measured every piece of the IOI mechanism on GPT-2 small directly:

1. **Duplicate / induction heads** (early; 5.5, 3.0 showed up in 01's patching) detect that one name repeated.
2. **S-inhibition heads** (7.3, 7.9, 8.6, 8.10) attend END→S2, read which name is the duplicated subject, and write a "don't attend here" signal — **big in patching, invisible in DLA**, because they act only on the movers.
3. **Name movers** (9.9, 9.6, 10.0) attend END→IO and copy it — **big in DLA**, the actual answer-writers. The S-inhibition signal is what aims them at Mary instead of John.
4. **Negative name movers** (10.7, 11.10) attend END→IO *too*, but write it with the opposite sign — **big and negative in DLA**, partially cancelling the answer.

`detect duplicate → inhibit the subject → move the remaining name → (a little counter-pressure)`. That's the Wang et al. circuit, recovered from forward passes and dot products — no weights read by hand, just interventions and projections.

## Exercise 5 — break it on purpose

A circuit you can break is one you understand. Three quick falsification/stress tests. Predict each, then run.

**(a) Ablate the top name mover 9.9 vs. ablate the negative mover 10.7.** In 01 you saw zeroing 9.9 on the *clean* prompt barely moved the logit diff (+3.17 → +3.10) — backup movers compensate. Confirm, and contrast with ablating the negative mover 10.7: removing an *anti*-answer head should make the diff go *up*.

**(b) How much redundancy is there?** Zero the four big positive movers (9.9, 9.6, 10.0, 10.10) together; then widen to the whole mover/backup family across L9–L11. Find out where redundancy actually runs out.

**Checkpoint:**

```
baseline:                              +3.17
ablate 9.9 (one mover):                +3.10   barely moves — backups cover it
ablate 10.7 (negative mover):          +4.53   goes UP — removing counter-pressure helps
ablate 4 movers 9.9/9.6/10.0/10.10:    +2.52   degrades but SURVIVES (not zero!)
ablate ~11 movers+backups (L9-11):     +0.62   redundancy finally exhausted
```

The textbook intuition says "kill the name movers and IOI dies." The measured reality is softer: knocking out the *four named* movers only drops the diff by ~0.65, because GPT-2's backup movers (the smaller L10–L11 heads) silently pick up the slack. You have to ablate the *whole family* to push the behavior toward zero. Report what you measure, not what the story predicts.

In [ ]:
def ablate_heads(tokens, heads):
    """Zero the listed (layer, head) z-outputs and return the logit diff."""
    from collections import defaultdict
    by_layer = defaultdict(list)
    for (L, H) in heads:
        by_layer[L].append(H)
    def make_hook(hs):
        def hook(act, hook):
            for h in hs:
                act[:, :, h, :] = 0.0
            return act
        return hook
    fwd = [(f"blocks.{L}.attn.hook_z", make_hook(hs)) for L, hs in by_layer.items()]
    return logit_diff(model.run_with_hooks(tokens, fwd_hooks=fwd))

movers4 = [(9,9), (9,6), (10,0), (10,10)]
# the broader name-mover / backup-mover family across L9-L11:
family  = [(9,9),(9,6),(9,0),(10,0),(10,1),(10,2),(10,6),(10,10),(11,2),(11,3),(11,9)]

print(f"baseline clean logit_diff:               {CLEAN_LD:+.3f}")
print(f"ablate 9.9 (one name mover):             {ablate_heads(clean_tokens, [(9,9)]):+.3f}")
print(f"ablate 10.7 (negative mover):            {ablate_heads(clean_tokens, [(10,7)]):+.3f}")
print(f"ablate 4 movers {movers4}: {ablate_heads(clean_tokens, movers4):+.3f}")
print(f"ablate full L9-11 mover family ({len(family)}):  {ablate_heads(clean_tokens, family):+.3f}")

<details><summary><b>Solution — Exercise 5</b></summary>

The `ablate_heads` helper above is the full solution — it groups heads by layer so each layer's `hook_z` gets a single hook zeroing all its targeted heads. Run the cell as written.

Reading the results:

- **9.9 alone → +3.10.** Barely a dent. Backup movers absorb it. A single load-bearing-*looking* head is not necessary in a redundant circuit.
- **10.7 alone → +4.53.** The logit diff *improves* when you remove a head — the unambiguous signature of an anti-answer component. If you only ever patched-in (never ablated), you might never realize the model is fighting itself.
- **4 movers → +2.52.** Drops, but the behavior **survives** — the four named movers are not the whole story. This is the honest correction to the textbook "kill the movers and it dies": GPT-2 spreads the copy across more heads than the four headline names.
- **Full L9–11 family → +0.62.** *Now* it's nearly dead. Redundancy is finite, but the floor is the whole mover/backup population, not the top four. This is exactly the "hydra" property from 01's 9.9-ablation sidebar, measured at family scale: the circuit regrows around any small cut and only collapses when you remove the redundancy itself.

The methodological keeper from 01, reinforced: **patch to find what carries signal, ablate to find what's necessary, and always try to break your own story.** DLA and attention told you *what* each head does; ablation tells you *how much the model actually depends on it* — and the answer, for any single head (or even the top four), is "less than the clean story implies."
</details>

---

**Where things stand.** Three notebooks in, the IOI circuit on GPT-2 small is no longer a 12×12 heatmap — it's a named mechanism with measured roles:

- **DLA** isolated the answer-writers (movers 9.9/9.6/10.0, negative movers 10.7/11.10).
- **Patching vs. DLA** separated heads that *act on the output* from heads that *act through other heads* (the S-inhibition family).
- **Attention patterns** confirmed the movers read the IO token and the S-inhibition heads read S2.
- **Ablation** showed the whole thing is redundant — robust to any single head, even to the top four movers, and only fragile when you remove the entire family.

**Next: `03_attention_patterns.ipynb`** — visualize the full attention maps, and use **path patching** to nail the S-inhibition → name-mover composition causally (the rigorous version of exercise 4B), tracing how the families actually wire into each other.